# pathoqc: BCSS-WSSS, aşama 1, temel model (seed 0)

Temel modeli (ResNet-50 + CAM) BCSS-WSSS'in yama etiketleriyle eğitir ve ürettiği kaba maskeleri
**doğrulama** kümesinde ölçer. Test kümesine dokunmaz.

Kod: https://github.com/suleymank12/pathoqc (aşağıda sabitlenen sürüm)

## 1. GPU açık mı?

GPU kapalıysa eğitim saatlerce sürer; o yüzden kapalıysa baştan durur.

In [ ]:
import torch

assert torch.cuda.is_available(), "GPU kapalı. Sağ panelde Accelerator olarak GPU seçilmeli."
print(torch.cuda.get_device_name(0), "| torch", torch.__version__)

## 2. Kodu GitHub'dan al

`checkout` satırı kodu belirli bir sürüme sabitler. Böylece bu sonucun hangi kodla alındığı hep bellidir.

In [ ]:
!mkdir -p /kaggle/tmp
!git clone -q https://github.com/suleymank12/pathoqc.git /kaggle/tmp/pathoqc
!git -C /kaggle/tmp/pathoqc checkout -q 2b4cc63
!git -C /kaggle/tmp/pathoqc log -1 --oneline
!pip install -q --no-deps -e /kaggle/tmp/pathoqc

## 3. Veriyi hazırla

Zip dosyası bu not defterine girdi olarak bağlıysa oradan alınır; bağlı değilse kaynağından indirilir.
İki durumda da dosyanın parmak izi (SHA-256) kontrol edilir: farklı bir dosyayla eğitim yapılamaz.

In [ ]:
import hashlib
import subprocess
import zipfile
from pathlib import Path

ZIP_ADI = "BCSS-WSSS.zip"
DRIVE_KIMLIGI = "1sCK8Cqsf4yCHzzIx98vitocES2L_tkjn"
BEKLENEN_SHA256 = "8cb54da91d60585c724ea765bfdd3ac9e70f4fdf96f5daaea865b2060ebed5d9"

bagli = sorted(Path("/kaggle/input").rglob(ZIP_ADI))
if bagli:
    zip_yolu = bagli[0]
    print("Veri girdi olarak bağlı:", zip_yolu)
else:
    zip_yolu = Path("/kaggle/tmp") / ZIP_ADI
    print("Veri bağlı değil, kaynağından indiriliyor")
    subprocess.run(["pip", "install", "-q", "-U", "gdown"], check=True)
    subprocess.run(["gdown", "-q", DRIVE_KIMLIGI, "-O", str(zip_yolu)], check=True)

ozet = hashlib.sha256()
with open(zip_yolu, "rb") as f:
    for parca in iter(lambda: f.read(1 << 20), b""):
        ozet.update(parca)
assert ozet.hexdigest() == BEKLENEN_SHA256, "Zip dosyası beklenen dosya değil"

with zipfile.ZipFile(zip_yolu) as z:
    z.extractall("/kaggle/tmp/veri")
print("Veri hazır:", sorted(p.name for p in Path("/kaggle/tmp/veri/BCSS-WSSS").iterdir()))

## 4. Eğit

Her epoch sonunda bir satır basılır: kayıp (loss), süre ve doğrulama kümesindeki ölçümler.

In [ ]:
!python /kaggle/tmp/pathoqc/scripts/train_stage1.py --dataset bcss --data-root /kaggle/tmp/veri --out-dir /kaggle/working/outputs --epochs 10 --batch-size 32 --workers 4 --seed 0

## 5. Sonuç

Eğitim yarıda kaldıysa bu hücre hata verir; yarım sonuç başarılı gibi görünmez.

In [ ]:
import json
from pathlib import Path

sonuc = json.loads(Path("/kaggle/working/outputs/bcss_stage1_seed0/history.json").read_text(encoding="utf-8"))
assert len(sonuc["history"]) == 10, "Eğitim tamamlanmadı"

for kayit in sonuc["history"]:
    print(f"epoch {kayit['epoch']:>2} | loss {kayit['loss']:.4f} | val mIoU {kayit['val']['miou']:.2f} | {kayit['val']['iou']}")
print("EN IYI VAL MIOU:", sonuc["best_val_miou"])